## 0. Download DREAMER from Kaggle (run once)

This uses the official Kaggle CLI to download and unzip [`phhasian0710/dreamer`](https://www.kaggle.com/datasets/phhasian0710/dreamer) into `.data`. The cell skips downloading when `.data/DREAMER.mat` already exists. If needed, install the CLI with `pip install kaggle` and configure a Kaggle API token first.

In [ ]:
from pathlib import Path
import shutil
import subprocess

DATA_DIR = Path('.data')
MAT_PATH = DATA_DIR / 'DREAMER.mat'
KAGGLE_DATASET = 'phhasian0710/dreamer'

if MAT_PATH.exists():
    print(f'Dataset already available: {MAT_PATH} ({MAT_PATH.stat().st_size / 1024**2:.1f} MiB)')
else:
    kaggle_executable = shutil.which('kaggle')
    if kaggle_executable is None:
        raise RuntimeError(
            'Kaggle CLI is not installed. Run `%pip install kaggle`, restart the kernel, '
            'and configure your Kaggle API token before retrying.'
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [kaggle_executable, 'datasets', 'download', KAGGLE_DATASET,
         '--path', str(DATA_DIR), '--unzip'],
        check=True,
    )
    # Normalize a nested archive layout to the path used by the converter.
    matches = list(DATA_DIR.rglob('DREAMER.mat'))
    if not matches:
        raise FileNotFoundError(f'Download finished, but DREAMER.mat was not found under {DATA_DIR}')
    if matches[0] != MAT_PATH:
        matches[0].replace(MAT_PATH)
    print(f'Dataset ready: {MAT_PATH} ({MAT_PATH.stat().st_size / 1024**2:.1f} MiB)')

# DREAMER CSV: memory-efficient overview and sample analysis

This notebook reads the partitioned files produced by `convert2excel.py`. It uses the small metadata tables for dataset-wide summaries and reads signal CSVs in chunks. The default analysis is intentionally limited to one subject/trial and 5,000 rows.

## 1. Convert the MAT file (run once)

Run the command below in a terminal from the project root. Conversion is streamed, but loading the 432 MiB MAT file still requires memory. Do not run conversion every time you open this notebook.

```powershell
python convert2excel.py --mat .data/DREAMER.mat --output data/dreamer_csv --resume
```

For a quick conversion test: `python convert2excel.py --subjects 1 --output data/dreamer_csv_test`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_DIR = Path('data/dreamer_csv')
CHUNK_SIZE = 50_000       # bounds temporary DataFrame memory
SAMPLE_ROWS = 5_000       # final in-memory analysis sample
SUBJECT = 1
TRIAL = 1
SIGNAL = 'EEG'            # EEG or ECG
CONDITION = 'stimuli'     # stimuli or baseline

required = [CSV_DIR / name for name in ('manifest.csv', 'subjects.csv', 'trials.csv')]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing conversion output: {missing}. Run the command above first.')

## 2. Dataset overview from metadata (no large signal files loaded)

In [ ]:
manifest = pd.read_csv(CSV_DIR / 'manifest.csv')
subjects = pd.read_csv(CSV_DIR / 'subjects.csv')
trials = pd.read_csv(CSV_DIR / 'trials.csv')

overview = pd.Series({
    'subjects': subjects['subject'].nunique(),
    'trials': len(trials),
    'signal CSV files': len(manifest),
    'signal rows': int(manifest['rows'].sum()),
    'CSV size (GiB)': round(manifest['size_bytes'].sum() / 1024**3, 3),
})
display(overview.to_frame('value'))
if overview['subjects'] < 23:
    print(f"NOTE: conversion is partial ({overview['subjects']}/23 subjects). Resume it with the command in section 1.")
display(manifest.groupby('signal').agg(files=('file', 'count'), rows=('rows', 'sum'),
                                         size_GiB=('size_bytes', lambda x: x.sum()/1024**3),
                                         sampling_rate_Hz=('sampling_rate_hz', 'first')).round(3))
display(subjects.head())

In [ ]:
# Ratings are trial-level, so their exact distribution is inexpensive to compute.
rating_counts = pd.concat(
    {name: trials[name].value_counts().sort_index() for name in ['valence', 'arousal', 'dominance']},
    axis=1
).fillna(0).astype(int)
display(rating_counts)
rating_counts.plot(kind='bar', figsize=(9, 4), title='DREAMER trial rating distributions')
plt.xlabel('Rating (1–5)'); plt.ylabel('Number of subject-trials'); plt.tight_layout(); plt.show()

## 3. Load only a bounded signal sample

`load_signal_sample` applies `usecols` and compact dtypes, filters each chunk immediately, and stops once enough rows are collected. Peak memory is approximately one chunk plus the requested sample.

In [ ]:
ID_DTYPES = {'subject': 'int8', 'trial': 'int8', 'condition': 'category',
             'time_idx': 'int32', 'time_sec': 'float32', 'valence': 'int8',
             'arousal': 'int8', 'dominance': 'int8'}

def load_signal_sample(csv_dir, subject=1, trial=1, signal='EEG', condition='stimuli',
                       max_rows=5_000, chunksize=50_000, channels=None):
    signal = signal.upper()
    path = Path(csv_dir) / f'subject_{subject:02d}_{signal.lower()}.csv'
    header = pd.read_csv(path, nrows=0).columns.tolist()
    available_channels = [name for name in header if name.startswith('ch_')]
    channels = available_channels if channels is None else channels
    invalid = set(channels) - set(available_channels)
    if invalid:
        raise ValueError(f'Channels not in {path.name}: {sorted(invalid)}')
    columns = list(ID_DTYPES) + channels
    dtypes = {**ID_DTYPES, **{name: 'float32' for name in channels}}
    pieces, remaining = [], max_rows
    for chunk in pd.read_csv(path, usecols=columns, dtype=dtypes, chunksize=chunksize):
        match = chunk[(chunk['trial'] == trial) & (chunk['condition'] == condition)]
        if not match.empty:
            pieces.append(match.iloc[:remaining].copy())
            remaining -= len(pieces[-1])
        if remaining <= 0:
            break
    if not pieces:
        raise ValueError('No rows matched the requested subject/trial/condition')
    return pd.concat(pieces, ignore_index=True)

sample = load_signal_sample(CSV_DIR, SUBJECT, TRIAL, SIGNAL, CONDITION,
                            SAMPLE_ROWS, CHUNK_SIZE)
print(f'Sample shape: {sample.shape}; memory: {sample.memory_usage(deep=True).sum()/1024**2:.2f} MiB')
display(sample.head())

## 4. Sample quality checks and descriptive analysis

In [ ]:
channel_cols = [name for name in sample if name.startswith('ch_')]
quality = pd.DataFrame({
    'missing': sample[channel_cols].isna().sum(),
    'infinite': np.isinf(sample[channel_cols]).sum(),
    'constant': sample[channel_cols].nunique(dropna=True).eq(1),
})
display(quality)
display(sample[channel_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

plot_channels = channel_cols[:min(4, len(channel_cols))]
ax = sample.plot(x='time_sec', y=plot_channels, figsize=(12, 5), linewidth=0.7,
                 title=f'{SIGNAL} sample — subject {SUBJECT}, trial {TRIAL}, {CONDITION}')
ax.set_xlabel('Time (seconds)'); ax.set_ylabel('Amplitude'); plt.tight_layout(); plt.show()

In [ ]:
# Correlation is computed only on the bounded sample.
corr = sample[channel_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(channel_cols)), channel_cols, rotation=90)
ax.set_yticks(range(len(channel_cols)), channel_cols)
ax.set_title(f'{SIGNAL} channel correlation (sample only)')
fig.colorbar(image, ax=ax, label='Pearson correlation')
plt.tight_layout(); plt.show()

## Practical notes

- Increase `SAMPLE_ROWS` only when needed; keep `CHUNK_SIZE` moderate.
- Select a subset of channels via `channels=['ch_1', 'ch_2']` to reduce parsing and memory.
- For model training, process one subject/trial at a time into windows or features and write those derived features to disk. Avoid concatenating all raw CSV files into one DataFrame.
- EEG is sampled at 128 Hz with 14 channels; ECG is sampled at 256 Hz with 2 channels. Ratings are integer valence, arousal, and dominance scores from 1 to 5.